# Fehlende Werte: Forward- und Backward-Fill

**Ziel:** Fehlende Satelliten-Features (Wolken, Orbitabdeckung) werden zeitlich
aufgefüllt. Pro `Place_ID` wird nach `Date` sortiert. Danach folgen zwei
**getrennte Schritte**:

1. **Forward-Fill:** der letzte bekannte Wert wird weitergetragen.
2. **Backward-Fill:** übrige Lücken (v. a. am Anfang der Zeitreihe) werden mit
   dem nächsten bekannten Wert gefüllt.

Wichtig:
- **0 gilt als fehlender Wert** (Platzhalter bei fehlender Satellitenmessung),
  **außer** in Spalten, in denen 0 ein realer Messwert sein kann
  (siehe Abschnitt 1). Dort bleibt 0 erhalten.
- Gefüllt wird **nur innerhalb eines Ortes**, nie über Orte hinweg.
- Train und Test werden **getrennt** gefüllt, damit keine Information vom
  einen Datensatz in den anderen fließt.
- Das Target und die Leakage-Spalten (`target_*`) werden nicht angefasst.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("data")
train = pd.read_csv(DATA_DIR / "Train.csv", parse_dates=["Date"])
test = pd.read_csv(DATA_DIR / "Test.csv", parse_dates=["Date"])

id_col = "Place_ID X Date"
meta_cols = [id_col, "Date", "Place_ID"]
features = [c for c in test.columns if c not in meta_cols]
print(f"Train: {train.shape}, Test: {test.shape}, Features: {len(features)}")

## 1. Spalten, in denen 0 ein realer Wert sein kann

- `*cloud_fraction`: 0 bedeutet „wolkenfrei“. Diese Spalten haben mehr Nullen
  als Platzhalter-Zeilen, die 0 ist hier also oft echt.
- Windkomponenten (`u_component_*`, `v_component_*`): 0 m/s ist möglich.

In allen anderen Spalten (Winkel, Höhen, Säulendichten, ...) ist exakt 0
ein Platzhalter und wird als fehlend behandelt.

In [ ]:
zero_ok = [c for c in features
           if "cloud_fraction" in c or "component_of_wind" in c]
zero_missing = [c for c in features if c not in zero_ok]
print(f"0 bleibt erhalten in {len(zero_ok)} Spalten:")
print(*zero_ok, sep="\n")
print(f"\n0 wird zu NaN in {len(zero_missing)} Spalten")

## 2. Fehlende Werte vorher

In [ ]:
def count_missing(df):
    """NaN in allen Features plus 0 in den Spalten, in denen 0 Platzhalter ist."""
    return int(df[features].isna().sum().sum()
               + (df[zero_missing] == 0).sum().sum())


before = {"train": count_missing(train), "test": count_missing(test)}
print("Fehlende Werte (NaN oder Platzhalter-0) vorher:", before)
train[features].isna().mean().mul(100).sort_values(ascending=False).head(10).round(1)

## 3. Vorbereitung: Platzhalter-0 zu NaN, Sortierung

In [ ]:
def prepare(df: pd.DataFrame) -> pd.DataFrame:
    """Sortiert nach Ort und Datum und setzt Platzhalter-0 auf NaN."""
    df = df.sort_values(["Place_ID", "Date"]).reset_index(drop=True)
    df[zero_missing] = df[zero_missing].replace(0, np.nan)
    return df


train_prep = prepare(train)
test_prep = prepare(test)
print("NaN nach Vorbereitung:",
      {"train": int(train_prep[features].isna().sum().sum()),
       "test": int(test_prep[features].isna().sum().sum())})

## 4. Forward-Fill (pro Ort)

In [ ]:
def forward_fill(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[features] = df.groupby("Place_ID")[features].ffill()
    return df


train_ffill = forward_fill(train_prep)
test_ffill = forward_fill(test_prep)
print("NaN nach Forward-Fill:",
      {"train": int(train_ffill[features].isna().sum().sum()),
       "test": int(test_ffill[features].isna().sum().sum())})

## 5. Backward-Fill (pro Ort)

Läuft auf dem Ergebnis des Forward-Fills und füllt nur noch die Lücken, die
der Forward-Fill nicht erreichen konnte (vor dem ersten bekannten Wert eines
Ortes).

In [ ]:
def backward_fill(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[features] = df.groupby("Place_ID")[features].bfill()
    return df


train_filled = backward_fill(train_ffill)
test_filled = backward_fill(test_ffill)
print("NaN nach Backward-Fill:",
      {"train": int(train_filled[features].isna().sum().sum()),
       "test": int(test_filled[features].isna().sum().sum())})

### Variante: nur Backward-Fill

Zum Vergleich wird der Backward-Fill auch **allein** auf den vorbereiteten Daten
ausgeführt (ohne vorherigen Forward-Fill). Jede Lücke wird dann mit dem
**nächsten** bekannten Wert des Ortes gefüllt, nicht mit dem letzten.

In [ ]:
train_bfill_only = backward_fill(train_prep)
test_bfill_only = backward_fill(test_prep)

summary = pd.DataFrame({
    "vorbereitet (NaN)": [train_prep[features].isna().sum().sum(),
                          test_prep[features].isna().sum().sum()],
    "nur Backward-Fill": [train_bfill_only[features].isna().sum().sum(),
                          test_bfill_only[features].isna().sum().sum()],
    "Forward + Backward": [train_filled[features].isna().sum().sum(),
                           test_filled[features].isna().sum().sum()],
}, index=["train", "test"])
summary

## 6. Kontrolle

In [ ]:
for name, df in [("train", train_filled), ("test", test_filled)]:
    rest = df[features].isna().sum()
    rest = rest[rest > 0]
    print(f"\n{name}: {len(rest)} Spalten mit Restlücken")
    if len(rest):
        display(rest.sort_values(ascending=False).head(10))

In [ ]:
# Sanity-Check: Zeilenzahl, Target und alle echten Werte unverändert
assert len(train_filled) == len(train) and len(test_filled) == len(test)
assert train_prep["target"].equals(train_filled["target"])
mask = train_prep[features].notna()
assert (train_prep[features].where(mask).fillna(0)
        == train_filled[features].where(mask).fillna(0)).all().all()
print("OK: Echte Werte und Target unverändert")

## 7. Hinweis zu Restlücken

Orte, für die ein Feature im gesamten Zeitraum nie gemessen wurde (v. a. CH4),
bleiben `NaN`, da es nichts zum Weitertragen gibt. Diese werden im Modell
separat behandelt (z. B. Baummodell mit `NaN`-Support oder Mittelwert-Imputation
mit Train-Mittelwerten).

In [ ]:
train_filled.to_csv(DATA_DIR / "Train_filled.csv", index=False)
test_filled.to_csv(DATA_DIR / "Test_filled.csv", index=False)